In [ ]:
import numpy as np

deg = {'SOFT': {'base': 90.0, 'd': 0.0844},
       'MEDIUM': {'base': 90.6, 'd': 0.0369},
       'HARD': {'base': 91.2, 'd': 0.0521}}
FUEL = -0.10
PIT_LOSS = 26.4          # green pit cost (measured)
PIT_SC   = 12.0          # pit cost under safety car (cited, Aguad)
N_LAPS   = 57
COMPOUNDS = ['SOFT', 'MEDIUM', 'HARD']

P_SC   = 0.0135          # your 2025 measurement: 54%/race -> ~1.35%/lap
SC_DUR = 4               # (legacy) unused; the chain now has 2 states
Q_END  = 0.25            # SC ends per lap (memoryless); E[dur]=1/q=4
SC_LAP_TIME = 140.0      # everyone crawls; same for all, so it doesn't distort choices

def lap_time(compound, tyre_age, lap_number):
    p = deg[compound]
    return p['base'] + p['d'] * tyre_age + FUEL * lap_number

In [ ]:
def flag_transition(f):
    """Returns list of (next_flag, probability). The Markov chain."""
    if f == 0:                       # green now
        return [(0, 1 - P_SC), (1, P_SC)]
    else:                            # under SC: ends w.p. Q_END, else continues
        return [(1, 1 - Q_END), (0, Q_END)]

# visualize it as a matrix
states = list(range(2))     # 0=green, 1=under safety car
T = np.zeros((len(states), len(states)))
for f in states:
    for nf, p in flag_transition(f):
        T[f, nf] = p
print("Flag transition matrix (rows=now, cols=next):")
print(np.round(T, 4))

In [40]:
B_MAX   = 4        # battery levels 0..4 (~1 MJ each, FIA 2026: 4 MJ usable)
DELTA   = 0.4      # deploy = -0.4s this lap; harvest = +0.4s (stylized; sweep later)
SC_RECHARGE = 1    # each SC lap refills 1 level for free (the 2026 coupling)
ENERGY_ACTIONS = ['deploy', 'hold', 'harvest']

In [ ]:
import numpy as np

def solve_sdp(battery=True):
    NC, NA, NM, NF, NB = 3, N_LAPS + 1, 2, 2, B_MAX + 1
    INF = float('inf')
    V   = np.full((N_LAPS + 2, NC, NA, NM, NF, NB), INF)
    PPOL = np.full((N_LAPS + 1, NC, NA, NM, NF, NB), -1, dtype=int)  # -1 stay, 0/1/2 pit-to
    EPOL = np.full((N_LAPS + 1, NC, NA, NM, NF, NB), 1,  dtype=int)  # 0 deploy,1 hold,2 harvest
    V[N_LAPS + 1, :, :, 1, :, :] = 0.0                # legal finishes

    energy_opts = [0, 1, 2] if battery else [1]        # OFF: hold only

    for n in range(N_LAPS, 0, -1):
        for ci in range(NC):
            c = COMPOUNDS[ci]
            for a in range(NA):
                for m in range(NM):
                    for f in range(NF):
                        for b in range(NB):
                            best, bp, be = INF, -1, 1
                            if f == 0:  # ---- GREEN ----
                                for e in energy_opts:
                                    if e == 0 and b == 0: continue        # can't deploy empty
                                    if e == 2 and b == B_MAX: continue    # can't harvest full
                                    dt = (-DELTA if e == 0 else DELTA if e == 2 else 0.0)
                                    b2 = b - 1 if e == 0 else b + 1 if e == 2 else b
                                    # stay
                                    if a + 1 < NA:
                                        cost = lap_time(c, a + 1, n) + dt
                                        ev = sum(p * V[n+1, ci, a+1, m, nf, b2]
                                                 for nf, p in flag_transition(f))
                                        if cost + ev < best: best, bp, be = cost + ev, -1, e
                                    # pit to x
                                    for xi in range(NC):
                                        m2 = 1 if (xi != ci or m == 1) else 0
                                        cost = lap_time(COMPOUNDS[xi], 1, n) + PIT_LOSS + dt
                                        ev = sum(p * V[n+1, xi, 1, m2, nf, b2]
                                                 for nf, p in flag_transition(f))
                                        if cost + ev < best: best, bp, be = cost + ev, xi, e
                            else:       # ---- SAFETY CAR ----
                                b2 = min(B_MAX, b + SC_RECHARGE) if battery else b
                                # stay (no wear, free recharge)
                                cost = SC_LAP_TIME
                                ev = sum(p * V[n+1, ci, a, m, nf, b2]
                                         for nf, p in flag_transition(f))
                                if cost + ev < best: best, bp, be = cost + ev, -1, 1
                                # pit cheap
                                for xi in range(NC):
                                    m2 = 1 if (xi != ci or m == 1) else 0
                                    cost = SC_LAP_TIME + PIT_SC
                                    ev = sum(p * V[n+1, xi, 0, m2, nf, b2]
                                             for nf, p in flag_transition(f))
                                    if cost + ev < best: best, bp, be = cost + ev, xi, 1
                            V[n, ci, a, m, f, b] = best
                            PPOL[n, ci, a, m, f, b] = bp
                            EPOL[n, ci, a, m, f, b] = be
        if n % 10 == 0: print(f"  [battery={'ON' if battery else 'OFF'}] solved to lap {n}")
    return V, PPOL, EPOL

print("Solving battery OFF (pre-2026)...")
V_off, P_off, E_off = solve_sdp(battery=False)
print("Solving battery ON (2026)...")
V_on,  P_on,  E_on  = solve_sdp(battery=True)
print("\nExpected time, SOFT start, half battery:")
print("  OFF:", round(V_off[1, 0, 0, 0, 0, 2], 1))
print("  ON: ", round(V_on [1, 0, 0, 0, 0, 2], 1))

In [42]:
# --- Stackelberg pit-window game (Aguad-style: leader commits, follower responds) ---
AGE_AT_WINDOW = 20
u   = deg['MEDIUM']['d'] * AGE_AT_WINDOW   # undercut gain = 0.74s (your calibration)
BEN = 0.30                                  # value of pitting later (fresher tires at stint end)
POS = 1.5                                   # penalty for finishing behind

def payoffs(a, b, g):
    """a,b: 0=pit NOW, 1=pit NEXT. Returns (A_cost, B_cost), lower=better."""
    shift = (u if (a==0 and b==1) else -u if (a==1 and b==0) else 0)
    final_gap = g + shift
    pos = (0.0, POS) if final_gap > 0 else (POS, 0.0) if final_gap < 0 else (POS/2, POS/2)
    return (pos[0] - (BEN if a==1 else 0), pos[1] - (BEN if b==1 else 0))

def stackelberg(g):
    """Leader A picks anticipating follower B's best response."""
    best = None
    for a in [0,1]:
        b_star = min([0,1], key=lambda b: payoffs(a,b,g)[1])   # B's response
        a_cost = payoffs(a, b_star, g)[0]
        if best is None or a_cost < best[0]:
            best = (a_cost, a, b_star)
    return ['NOW','NEXT'][best[1]], ['NOW','NEXT'][best[2]]

print(f"u = {u:.2f}s | leader's move by gap (Stackelberg):")
print(f"{'gap g':>6s} | {'undercut live?':>14s} | {'A (leader)':>10s} | {'B (follower)':>12s}")
print("-" * 55)
for g in [0.2, 0.4, 0.6, 0.74, 0.9, 1.2, 2.0, 3.0]:
    A_move, B_move = stackelberg(g)
    print(f"{g:>6.2f} | {('YES' if g < u else 'no'):>14s} | {A_move:>10s} | {B_move:>12s}")

u = 0.74s | leader's move by gap (Stackelberg):
 gap g | undercut live? | A (leader) | B (follower)
-------------------------------------------------------
  0.20 |            YES |        NOW |         NEXT
  0.40 |            YES |        NOW |         NEXT
  0.60 |            YES |        NOW |         NEXT
  0.74 |             no |       NEXT |         NEXT
  0.90 |             no |       NEXT |         NEXT
  1.20 |             no |       NEXT |         NEXT
  2.00 |             no |       NEXT |         NEXT
  3.00 |             no |       NEXT |         NEXT


In [43]:
g = 1.0
M = build_game(T_pit_on, T_stay_on, g, u)
print(f"Payoff matrix at g={g} (battery ON). Format: (A's time, B's time). Lower = better.\n")
print(f"{'':12s} B: STAY{'':10s} B: PIT")
for a in [0,1]:
    row = ['STAY','PIT '][a]
    print(f"A: {row}   {M[(a,0)][0]:7.1f},{M[(a,0)][1]:7.1f}   {M[(a,1)][0]:7.1f},{M[(a,1)][1]:7.1f}")
print(f"\nNash equilibrium: {nash(M)}")

Payoff matrix at g=1.0 (battery ON). Format: (A's time, B's time). Lower = better.

             B: STAY           B: PIT
A: STAY    2974.8, 2976.3    2974.8, 2981.5
A: PIT     2980.0, 2976.3    2980.0, 2981.5

Nash equilibrium: [('NOW', 'NOW')]


In [44]:
print(f"T_pit_on  = {T_pit_on:.1f}")
print(f"T_stay_on = {T_stay_on:.1f}")
print(f"Pit penalty vs staying: {T_pit_on - T_stay_on:+.1f}s")

T_pit_on  = 2980.0
T_stay_on = 2974.8
Pit penalty vs staying: +5.2s
